# ProofAgent™ — Log-based (local notebook)

Same naming as **`examples/judge_led_quickstart.py`**: **`tested_agent_config`**, **`your_agent`**, **`byo`**, **`pa`**, **`result`**, plus **`logs`** for replay.

**Role & tools** align with the LangGraph IT-ops example: short **`role`** slug (e.g. **`customer_support`**) must match your project; long **`description`** is metadata. Your production agent may be a graph; here you submit **recorded** turns.

```bash
pip install proofagent-sdk
```

Requires a **Log-Based** project key **`PROOFAGENT_API_KEY`**. Optional **`OPENAI_API_KEY`**.

In [1]:
import os
from typing import Any

from proofagent import ProofAgent, TestedAgent, print_full_evaluation_report

if not os.environ.get("PROOFAGENT_API_KEY", "").strip():
    raise RuntimeError("Set PROOFAGENT_API_KEY (Log-Based project).")

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

logs: list[dict[str, Any]] = [
    {"turn_index": 1, "user_message": "Hi", "agent_answer": "Hello."},
    {"turn_index": 2, "user_message": "Status?", "agent_answer": "On it."},
]

tested_agent_config = {
    "role": "customer_support",
    "description": (
        "IT operations and support agent: triages incidents, looks up ticket status, and answers user "
        "questions with concise, policy-aware replies. Aligns with the LangGraph ReAct IT-ops example."
    ),
    "tools": [
        {"name": "ticket_stub", "description": "Look up ticket status (demo / ITSM integration)."},
    ],
}

your_agent = TestedAgent.from_json(tested_agent_config)

byo = os.environ.get("OPENAI_API_KEY", "").strip()


async def run():
    async with ProofAgent.from_env(
        reasoning_provider="openai" if byo else None,
        reasoning_model=OPENAI_MODEL if byo else None,
    ) as pa:
        return await pa.evaluate_logs(
            logs,
            your_agent,
            verbose=True,
            poll_complete_timeout=900.0,
        )


result = await run()
print(result.label, result.score, result.run_id)
print("\n--- Run ID ---\n", result.run_id)
print_full_evaluation_report(result.report)

ProofAgentError: Log-Based Evaluation requires a ProofAgent project whose evaluation mode is log_replay, context_eval, or multi_log. Your project 'Air-bot' has mode 'judge_led'. Create a Log-Based project in the ProofAgent dashboard and use that project's API key.